# Notebook-first application walkthrough

**Problem / objective:** Forecast demand with chronological validation so operations can prepare capacity without leaking future information.

**Decision / solution:** Translate forecast errors into staffing/rebalancing risk and identify the hours or weather conditions where extra capacity is most valuable.

This front section is intentionally analysis-first. It uses direct notebook code for inspection, EDA, visualisation and evidence review. The original notebook work is preserved below, followed by modular production code where that adds engineering evidence.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
PROJECT_SLUG = 'xgboost_bike_demand'
ROOT = Path.cwd()
if not (ROOT / 'projects').exists():
    candidate = ROOT.parent.parent if ROOT.name == PROJECT_SLUG else ROOT
    if (candidate / 'projects').exists():
        ROOT = candidate
PROJECT = ROOT / 'projects' / PROJECT_SLUG
if not PROJECT.exists() and Path.cwd().name == PROJECT_SLUG:
    PROJECT = Path.cwd()
    ROOT = PROJECT.parent.parent
assert PROJECT.exists(), f'Project directory not found: {PROJECT}'
print('Repository root:', ROOT.resolve())
print('Project:', PROJECT.resolve())


## 1. Find the real data and retained evidence

Instead of hiding the dataset behind a helper function, start by seeing what the project actually ships: raw/small data, fixtures, outputs, results and verified evidence. External large datasets remain reproducibly downloadable from the documented source.


In [ ]:
candidate_files = []
for pattern in ('*.csv', '*.parquet', '*.json', '*.tsv', '*.txt'):
    candidate_files.extend(PROJECT.rglob(pattern))
verified_dir = ROOT / 'verified' / PROJECT_SLUG
if verified_dir.exists():
    for pattern in ('*.csv', '*.parquet', '*.json', '*.tsv', '*.txt'):
        candidate_files.extend(verified_dir.rglob(pattern))
candidate_files = sorted({p.resolve() for p in candidate_files if p.is_file()})
file_inventory = pd.DataFrame({
    'file': [str(p.relative_to(ROOT)) if ROOT in p.parents else str(p) for p in candidate_files],
    'suffix': [p.suffix.lower() for p in candidate_files],
    'size_kb': [round(p.stat().st_size / 1024, 1) for p in candidate_files],
})
display(file_inventory.head(40))
print(f'Inspectable local data/evidence files: {len(file_inventory):,}')


## 2. Direct tabular data audit

The code below deliberately avoids a project-specific wrapper. It opens the first sensible local tabular asset, shows its schema and quality profile, and makes the data issues visible before modelling. If the full raw dataset is external, run the project's documented download cell/entry point first and rerun this section.


In [ ]:
tabular_candidates = [p for p in candidate_files if p.suffix.lower() in {'.csv', '.tsv', '.parquet'}]
preferred = [p for p in tabular_candidates if not any(token in p.name.lower() for token in ('metric', 'summary', 'verification'))]
tabular_path = (preferred or tabular_candidates or [None])[0]
df = None
if tabular_path is not None:
    if tabular_path.suffix.lower() == '.parquet':
        df = pd.read_parquet(tabular_path)
    else:
        sep = '\t' if tabular_path.suffix.lower() == '.tsv' else ','
        df = pd.read_csv(tabular_path, sep=sep, nrows=200_000)
    print('Loaded:', tabular_path)
    print('Shape:', df.shape)
    display(df.head())
    audit = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'missing': df.isna().sum(),
        'missing_pct': (100 * df.isna().mean()).round(2),
        'unique': df.nunique(dropna=False),
    }).sort_values(['missing_pct', 'unique'], ascending=[False, False])
    display(audit.head(30))
    print('Duplicate rows:', int(df.duplicated().sum()))
else:
    print('No local CSV/TSV/Parquet found yet. Use the project README/run path to download or build the documented dataset, then rerun this audit.')


## 3. Exploratory data analysis and visualisation

These plots are intentionally created in the notebook rather than described in prose. They expose distribution, missingness, scale, category balance and numeric relationships before any final model decision.


In [ ]:
if df is not None and len(df):
    missing_pct = (100 * df.isna().mean()).sort_values(ascending=False).head(20)
    missing_pct = missing_pct[missing_pct > 0]
    if len(missing_pct):
        plt.figure(figsize=(10, 4))
        missing_pct.plot(kind='bar')
        plt.title('Missing values by feature (%)')
        plt.ylabel('Missing %')
        plt.xticks(rotation=60, ha='right')
        plt.tight_layout()
        plt.show()

    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()[:8]
    for col in numeric_cols:
        series = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(series):
            plt.figure(figsize=(8, 4))
            plt.hist(series, bins=30, alpha=0.8)
            plt.axvline(series.median(), linestyle='--', label=f'median={series.median():.2f}')
            plt.title(f'Distribution: {col}')
            plt.xlabel(col)
            plt.ylabel('Count')
            plt.legend()
            plt.tight_layout()
            plt.show()

    categorical_cols = [c for c in df.columns if c not in numeric_cols and df[c].nunique(dropna=False) <= 30][:4]
    for col in categorical_cols:
        counts = df[col].fillna('<missing>').astype(str).value_counts().head(15)
        plt.figure(figsize=(9, 4))
        counts.sort_values().plot(kind='barh')
        plt.title(f'Top categories: {col}')
        plt.xlabel('Rows')
        plt.tight_layout()
        plt.show()

    if len(numeric_cols) >= 2:
        corr = df[numeric_cols].corr(numeric_only=True)
        plt.figure(figsize=(8, 6))
        image = plt.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
        plt.colorbar(image, label='Correlation')
        plt.xticks(range(len(corr.columns)), corr.columns, rotation=60, ha='right')
        plt.yticks(range(len(corr.index)), corr.index)
        plt.title('Numeric correlation matrix')
        plt.tight_layout()
        plt.show()

    if len(numeric_cols) >= 2:
        x_col, y_col = numeric_cols[0], numeric_cols[-1]
        sample = df[[x_col, y_col]].dropna().sample(min(3000, len(df.dropna(subset=[x_col, y_col]))), random_state=42)
        if len(sample):
            plt.figure(figsize=(7, 5))
            plt.scatter(sample[x_col], sample[y_col], alpha=0.35, s=18)
            plt.xlabel(x_col)
            plt.ylabel(y_col)
            plt.title(f'{y_col} versus {x_col}')
            plt.tight_layout()
            plt.show()
else:
    print('Run the documented data-build/download path, then rerun this section to render raw-data EDA.')


## 4. Inspect the measured results, not just the code

A portfolio project is stronger when it retains evidence. This section reads machine-readable JSON/CSV outputs and turns scalar metrics into a quick visual comparison.


In [ ]:
json_files = [p for p in candidate_files if p.suffix.lower() == '.json']
metric_rows = []
for path in json_files[:30]:
    try:
        payload = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        continue
    stack = [('', payload)]
    while stack:
        prefix, value = stack.pop()
        if isinstance(value, dict):
            for key, child in value.items():
                stack.append((f'{prefix}.{key}' if prefix else str(key), child))
        elif isinstance(value, (int, float)) and not isinstance(value, bool) and np.isfinite(value):
            metric_rows.append({
                'file': str(path.relative_to(ROOT)) if ROOT in path.parents else str(path),
                'metric': prefix,
                'value': float(value),
            })
metrics_df = pd.DataFrame(metric_rows)
if len(metrics_df):
    display(metrics_df.head(40))
    plot_df = metrics_df[np.isfinite(metrics_df['value'])].copy()
    plot_df = plot_df[plot_df['value'].abs() < 1_000_000].head(20)
    if len(plot_df):
        labels = (plot_df['file'].str.split('/').str[-1] + ' :: ' + plot_df['metric']).tolist()
        plt.figure(figsize=(10, max(4, 0.35 * len(plot_df))))
        plt.barh(range(len(plot_df)), plot_df['value'])
        plt.yticks(range(len(plot_df)), labels)
        plt.title('Retained project metrics / evidence')
        plt.tight_layout()
        plt.show()
else:
    print('No scalar JSON evidence found. Run the project and retain metrics/results before treating it as complete.')


## 5. Reproduce the application

The notebook should be understandable without running anything, but a reviewer can reproduce the canonical application below. The switch is off by default so opening the notebook never triggers a long training job unexpectedly.


In [ ]:
RUN_PROJECT = False
entrypoint = PROJECT / 'run.py'
if RUN_PROJECT and entrypoint.exists():
    subprocess.run([sys.executable, str(entrypoint)], cwd=PROJECT, check=True)
elif entrypoint.exists():
    print(f'Reproduce with: cd {PROJECT} && {sys.executable} run.py')
else:
    print('This project uses a different documented entry point; see README.md in the project folder.')


## 6. Decision / solution

Translate forecast errors into staffing/rebalancing risk and identify the hours or weather conditions where extra capacity is most valuable.

The final recommendation should be tied to the measured validation evidence and error analysis below. A model is not the solution by itself; the solution is the decision process built around it.


# XGBoost Bike Demand — Capacity Planning Application

## Problem and objective
Forecast hourly bike-rental demand with a time-aware XGBoost workflow and convert predictions into operational demand bands for rebalancing and staffing decisions.


## Dataset and provenance
UCI Bike Sharing Dataset (`hour.csv`). The application downloads the public UCI archive reproducibly. Leakage-prone component counts (`casual` and `registered`) are explicitly removed because they sum to the target.


In [ ]:
from run import load_data, audit_data, add_features, chronological_split, feature_columns
raw = load_data()
audit_data(raw)


## Modelling, analysis and validation
The project uses cyclic calendar features, a strict chronological train/validation/test boundary, a seasonal hour-of-week baseline, XGBoost hyperparameter search, MAE/RMSE/RMSLE/R², error slices, feature importance, model persistence and an operations decision layer. The complete canonical code is mirrored below by the portfolio workflow.


In [ ]:
data = add_features(raw)
train, validation, test, split = chronological_split(data)
features = feature_columns(data)
split, features[:12]


In [ ]:
from run import seasonal_hour_baseline, metrics
baseline_prediction = seasonal_hour_baseline(train, validation)
metrics(validation['cnt'], baseline_prediction)


## Results, limitations and next steps
Metrics, tuning trials, feature importance, segment error analysis and test predictions are saved under `results/`. The operational thresholds and residual uncertainty range are illustrative. Production use requires live inventory, current weather forecasts, rolling validation, calibrated uncertainty and explicit capacity costs.

## Reproducibility
Run `python run.py`. Tests are in `tests/test_xgboost.py`; the trained model is written to `artifacts/`.


## Interview discussion
Be ready to explain why a chronological split is essential, why `casual` and `registered` are target leakage, what XGBoost adds over the seasonal baseline, how tree depth/learning rate/regularisation interact, how you diagnose segment errors, and why operational thresholds need real capacity costs.


# Deeper exploratory analysis and retained evidence

These direct notebook cells extend the initial EDA with data-quality, scale, relationship, output and error diagnostics. They are intentionally visible here rather than hidden behind project helper functions.


In [ ]:
# Extended data-quality scorecard
if df is not None and len(df):
    quality_rows = []
    for col in df.columns:
        series = df[col]
        row = {
            'feature': col,
            'dtype': str(series.dtype),
            'rows': len(series),
            'missing': int(series.isna().sum()),
            'missing_pct': float(100 * series.isna().mean()),
            'unique': int(series.nunique(dropna=False)),
            'unique_pct': float(100 * series.nunique(dropna=False) / max(len(series), 1)),
        }
        if pd.api.types.is_numeric_dtype(series):
            values = pd.to_numeric(series, errors='coerce').dropna()
            if len(values):
                q1, q3 = values.quantile([0.25, 0.75])
                iqr = q3 - q1
                row.update({
                    'mean': float(values.mean()),
                    'median': float(values.median()),
                    'std': float(values.std()),
                    'p05': float(values.quantile(0.05)),
                    'p95': float(values.quantile(0.95)),
                    'skew': float(values.skew()),
                    'iqr_outliers': int(((values < q1 - 1.5*iqr) | (values > q3 + 1.5*iqr)).sum()),
                })
        quality_rows.append(row)
    deep_quality = pd.DataFrame(quality_rows)
    display(deep_quality.sort_values(['missing_pct','unique'], ascending=[False,False]).head(40))
    if 'iqr_outliers' in deep_quality:
        outlier_view = deep_quality.dropna(subset=['iqr_outliers']).sort_values('iqr_outliers', ascending=False).head(15)
        if len(outlier_view):
            plt.figure(figsize=(10,4))
            plt.bar(outlier_view['feature'], outlier_view['iqr_outliers'])
            plt.title('Potential IQR outliers by feature')
            plt.ylabel('Rows')
            plt.xticks(rotation=60, ha='right')
            plt.tight_layout()
            plt.show()
    card = deep_quality.sort_values('unique', ascending=False).head(20)
    plt.figure(figsize=(10,4))
    plt.bar(card['feature'], card['unique'])
    plt.title('Feature cardinality')
    plt.ylabel('Unique values')
    plt.xticks(rotation=60, ha='right')
    plt.tight_layout()
    plt.show()
    print('Constant columns:', deep_quality.loc[deep_quality['unique'] <= 1, 'feature'].tolist())
    print('High-missing columns:', deep_quality.loc[deep_quality['missing_pct'] >= 30, 'feature'].tolist())
    print('Possible identifier columns:', deep_quality.loc[deep_quality['unique_pct'] >= 95, 'feature'].tolist()[:20])
else:
    print('Materialise the documented dataset to run the extended data-quality scorecard.')


In [ ]:
# Numeric distributions, spread and strongest pairwise relationships
if df is not None and len(df):
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()[:12]
    for col in numeric_cols:
        values = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(values) < 5:
            continue
        clipped = values.clip(values.quantile(0.01), values.quantile(0.99))
        plt.figure(figsize=(8,4))
        plt.hist(clipped, bins=35, alpha=0.82)
        plt.axvline(values.median(), linestyle='--', label=f'median={values.median():.3g}')
        plt.axvline(values.mean(), linestyle=':', label=f'mean={values.mean():.3g}')
        plt.title(f'Distribution: {col} (1st–99th percentile)')
        plt.xlabel(col)
        plt.ylabel('Rows')
        plt.legend()
        plt.tight_layout()
        plt.show()
        plt.figure(figsize=(8,3))
        plt.boxplot(values, vert=False, showfliers=True)
        plt.title(f'Spread / outliers: {col}')
        plt.xlabel(col)
        plt.tight_layout()
        plt.show()
    if len(numeric_cols) >= 2:
        corr = df[numeric_cols].corr(numeric_only=True)
        pairs = []
        for i, left in enumerate(corr.columns):
            for right in corr.columns[i+1:]:
                value = corr.loc[left, right]
                if pd.notna(value):
                    pairs.append({'feature_a': left, 'feature_b': right, 'correlation': float(value), 'abs_correlation': float(abs(value))})
        corr_pairs = pd.DataFrame(pairs).sort_values('abs_correlation', ascending=False) if pairs else pd.DataFrame()
        if len(corr_pairs):
            display(corr_pairs.head(20).round(4))
            for _, pair in corr_pairs.head(4).iterrows():
                sample = df[[pair['feature_a'], pair['feature_b']]].dropna()
                if len(sample) > 3000:
                    sample = sample.sample(3000, random_state=42)
                plt.figure(figsize=(7,5))
                plt.scatter(sample[pair['feature_a']], sample[pair['feature_b']], alpha=0.30, s=16)
                plt.xlabel(pair['feature_a'])
                plt.ylabel(pair['feature_b'])
                plt.title(f"{pair['feature_a']} vs {pair['feature_b']} (r={pair['correlation']:.2f})")
                plt.tight_layout()
                plt.show()
    categorical = [c for c in df.columns if 2 <= df[c].nunique(dropna=False) <= 20][:8]
    for col in categorical:
        counts = df[col].fillna('<missing>').astype(str).value_counts().head(20)
        shares = 100 * counts / counts.sum()
        display(pd.DataFrame({'rows': counts, 'share_pct': shares.round(2)}))
        plt.figure(figsize=(8,4))
        counts.sort_values().plot(kind='barh')
        plt.title(f'Category balance: {col}')
        plt.xlabel('Rows')
        plt.tight_layout()
        plt.show()
else:
    print('Materialise the documented dataset to run distribution diagnostics.')


In [ ]:
# Temporal coverage where date/time fields exist
if df is not None and len(df):
    time_cols = [c for c in df.columns if any(token in str(c).lower() for token in ('date','time','timestamp','datetime'))]
    print('Date/time candidates:', time_cols[:10])
    for col in time_cols[:4]:
        converted = pd.to_datetime(df[col], errors='coerce')
        valid = converted.dropna()
        if len(valid) >= max(10, int(0.25*len(df))):
            print(col, 'range:', valid.min(), '→', valid.max())
            monthly = valid.dt.to_period('M').value_counts().sort_index()
            if len(monthly) > 1:
                plt.figure(figsize=(10,4))
                plt.plot(monthly.index.astype(str), monthly.values, marker='o')
                plt.title(f'Rows over time: {col}')
                plt.ylabel('Rows')
                plt.xticks(rotation=70, ha='right')
                plt.tight_layout()
                plt.show()


## Retained outputs and error analysis

A strong portfolio keeps inspectable evidence. The cells below profile compact result tables and automatically detect prediction-like columns for residual or misclassification analysis.


In [ ]:
# Load compact result/evidence tables
result_tables = []
for base in [PROJECT/'results', PROJECT/'outputs', PROJECT/'artifacts', ROOT/'verified'/PROJECT_SLUG]:
    if not base.exists():
        continue
    for path in sorted(base.rglob('*')):
        if path.is_file() and path.suffix.lower() in {'.csv','.tsv','.parquet'} and path.stat().st_size < 25_000_000:
            try:
                if path.suffix.lower() == '.parquet':
                    table = pd.read_parquet(path)
                else:
                    table = pd.read_csv(path, sep='	' if path.suffix.lower() == '.tsv' else ',')
            except Exception as exc:
                print('Could not read', path.name, '-', exc)
                continue
            result_tables.append((path, table))
            print('
RESULT TABLE:', path.relative_to(ROOT) if ROOT in path.parents else path)
            print('shape=', table.shape)
            display(table.head(15))
            numeric = table.select_dtypes(include=np.number).columns.tolist()[:12]
            if numeric:
                display(table[numeric].describe().T.round(4))
print('Inspectable result tables:', len(result_tables))


In [ ]:
# Automatic regression/classification-style error diagnostics
actual_tokens = ('actual','target','truth','y_true','observed','label')
pred_tokens = ('prediction','predicted','forecast','y_pred')
confidence_tokens = ('confidence','probability','proba','risk','uncertainty')
for path, table in result_tables:
    actual_cols = [c for c in table.columns if any(token in str(c).lower() for token in actual_tokens)]
    pred_cols = [c for c in table.columns if any(token in str(c).lower() for token in pred_tokens)]
    conf_cols = [c for c in table.columns if any(token in str(c).lower() for token in confidence_tokens)]
    if actual_cols and pred_cols and len(table):
        actual_col = actual_cols[0]
        pred_col = next((c for c in pred_cols if c != actual_col), pred_cols[0])
        actual_num = pd.to_numeric(table[actual_col], errors='coerce')
        pred_num = pd.to_numeric(table[pred_col], errors='coerce')
        numeric_mask = actual_num.notna() & pred_num.notna()
        if numeric_mask.sum() >= 10:
            residual = actual_num[numeric_mask] - pred_num[numeric_mask]
            abs_error = residual.abs()
            print('
', path.name, '| MAE=', round(float(abs_error.mean()),5), '| RMSE=', round(float(np.sqrt(np.mean(residual**2))),5), '| bias=', round(float(residual.mean()),5))
            plt.figure(figsize=(7,5))
            plt.scatter(actual_num[numeric_mask], pred_num[numeric_mask], alpha=0.35, s=18)
            lo = min(actual_num[numeric_mask].min(), pred_num[numeric_mask].min())
            hi = max(actual_num[numeric_mask].max(), pred_num[numeric_mask].max())
            plt.plot([lo,hi],[lo,hi], linestyle='--')
            plt.xlabel(str(actual_col))
            plt.ylabel(str(pred_col))
            plt.title(f'Actual vs predicted — {path.name}')
            plt.tight_layout()
            plt.show()
            plt.figure(figsize=(7,4))
            plt.hist(residual, bins=30, alpha=0.82)
            plt.axvline(0, linestyle='--')
            plt.title(f'Residual distribution — {path.name}')
            plt.tight_layout()
            plt.show()
            worst_idx = abs_error.nlargest(min(15,len(abs_error))).index
            cols = list(dict.fromkeys([actual_col,pred_col]+conf_cols[:2]))
            worst = table.loc[worst_idx, cols].copy()
            worst['absolute_error'] = abs_error.loc[worst_idx].values
            display(worst.sort_values('absolute_error', ascending=False))
        else:
            agreement = table[actual_col].astype(str) == table[pred_col].astype(str)
            print('
', path.name, '| classification agreement=', round(float(agreement.mean()),4))
            if (~agreement).any():
                display(table.loc[~agreement, [actual_col,pred_col]+conf_cols[:2]].head(20))
    elif conf_cols:
        for col in conf_cols[:2]:
            values = pd.to_numeric(table[col], errors='coerce').dropna()
            if len(values) >= 10:
                plt.figure(figsize=(7,4))
                plt.hist(values, bins=30, alpha=0.82)
                plt.title(f'{col} distribution — {path.name}')
                plt.tight_layout()
                plt.show()


In [ ]:
# Display retained visual evidence from actual project runs
png_files = []
for base in [PROJECT/'results', PROJECT/'outputs', PROJECT/'artifacts', ROOT/'verified'/PROJECT_SLUG]:
    if base.exists():
        png_files.extend(sorted(base.rglob('*.png')))
print('Retained PNG figures:', len(png_files))
for path in png_files[:12]:
    try:
        image = plt.imread(path)
        plt.figure(figsize=(10,6))
        plt.imshow(image)
        plt.axis('off')
        plt.title(str(path.relative_to(ROOT)) if ROOT in path.parents else path.name)
        plt.tight_layout()
        plt.show()
    except Exception as exc:
        print('Could not display', path.name, '-', exc)


In [ ]:
# Reproducibility and evidence checklist
checks = [
    {'check':'README present', 'status':(PROJECT/'README.md').exists()},
    {'check':'Recruiter notebook present', 'status':(PROJECT/'project_notebook.ipynb').exists()},
    {'check':'Python implementation present', 'status':any(PROJECT.rglob('*.py'))},
    {'check':'Tests present', 'status':(PROJECT/'tests').exists() and any((PROJECT/'tests').rglob('test*.py'))},
    {'check':'Result/evidence files present', 'status':bool(candidate_files)},
    {'check':'Machine-readable JSON evidence', 'status':bool(json_files)},
    {'check':'Retained visual evidence', 'status':bool(png_files)},
]
checklist = pd.DataFrame(checks)
display(checklist)
print('Evidence checklist pass rate:', f"{100*checklist['status'].mean():.1f}%")
print('A failed item is a prompt to strengthen the project, not something to hide.')


# Robustness, slices and decision analysis

A model or pipeline is useful only when we know where it works, where it fails and what action follows. This section adds direct slice analysis, sensitivity checks and a compact decision memo from the evidence already produced by the project.


In [ ]:
# Quantile slices for important numeric variables
if df is not None and len(df):
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()[:10]
    quantile_rows = []
    for col in numeric_cols:
        values = pd.to_numeric(df[col], errors='coerce')
        valid = values.dropna()
        if len(valid) < 20 or valid.nunique() < 5:
            continue
        quantiles = valid.quantile([0.01,0.05,0.10,0.25,0.50,0.75,0.90,0.95,0.99])
        for q, value in quantiles.items():
            quantile_rows.append({'feature':col, 'quantile':q, 'value':float(value)})
    quantile_table = pd.DataFrame(quantile_rows)
    if len(quantile_table):
        display(quantile_table.pivot(index='feature', columns='quantile', values='value').round(4))
        for col in quantile_table['feature'].unique()[:6]:
            view = quantile_table[quantile_table['feature']==col]
            plt.figure(figsize=(7,4))
            plt.plot(view['quantile'], view['value'], marker='o')
            plt.xlabel('Quantile')
            plt.ylabel(col)
            plt.title(f'Quantile profile: {col}')
            plt.tight_layout()
            plt.show()
else:
    print('Quantile slices become available after the project dataset is materialised.')


In [ ]:
# Missingness and duplication sensitivity
if df is not None and len(df):
    missing_by_row = df.isna().sum(axis=1)
    print('Rows with any missing value:', int((missing_by_row>0).sum()))
    print('Rows with 2+ missing values:', int((missing_by_row>=2).sum()))
    print('Exact duplicate rows:', int(df.duplicated().sum()))
    if missing_by_row.max() > 0:
        plt.figure(figsize=(7,4))
        missing_by_row.value_counts().sort_index().plot(kind='bar')
        plt.title('Missing cells per row')
        plt.xlabel('Missing cells')
        plt.ylabel('Rows')
        plt.tight_layout()
        plt.show()
    duplicated = df.duplicated(keep=False)
    if duplicated.any():
        display(df.loc[duplicated].head(20))
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()[:10]
    robust_rows = []
    for col in numeric_cols:
        values = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(values) < 20:
            continue
        median = values.median()
        mad = np.median(np.abs(values-median))
        robust_z = 0.6745*(values-median)/(mad if mad else 1.0)
        robust_rows.append({'feature':col, 'median':median, 'mad':mad, 'robust_outliers_abs_z_gt_3_5':int((np.abs(robust_z)>3.5).sum())})
    robust_outliers = pd.DataFrame(robust_rows).sort_values('robust_outliers_abs_z_gt_3_5', ascending=False) if robust_rows else pd.DataFrame()
    if len(robust_outliers):
        display(robust_outliers.round(4))


In [ ]:
# Concentration / imbalance analysis for important categorical dimensions
if df is not None and len(df):
    categorical = [c for c in df.columns if 2 <= df[c].nunique(dropna=False) <= 50][:10]
    concentration_rows = []
    for col in categorical:
        counts = df[col].fillna('<missing>').astype(str).value_counts()
        shares = counts / counts.sum()
        hhi = float((shares**2).sum())
        concentration_rows.append({'feature':col, 'categories':len(counts), 'largest_share':float(shares.iloc[0]), 'top3_share':float(shares.head(3).sum()), 'hhi':hhi})
    concentration = pd.DataFrame(concentration_rows).sort_values('hhi', ascending=False) if concentration_rows else pd.DataFrame()
    if len(concentration):
        display(concentration.round(4))
        plt.figure(figsize=(9,4))
        plt.bar(concentration['feature'], concentration['largest_share'])
        plt.ylabel('Largest category share')
        plt.title('Category concentration / imbalance')
        plt.xticks(rotation=60, ha='right')
        plt.tight_layout()
        plt.show()


In [ ]:
# Rank all retained scalar metrics and highlight likely success/risk signals
metric_records = []
for path in json_files[:60]:
    try:
        payload = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        continue
    stack = [('', payload)]
    while stack:
        prefix, value = stack.pop()
        if isinstance(value, dict):
            for key, child in value.items():
                stack.append((f'{prefix}.{key}' if prefix else str(key), child))
        elif isinstance(value, (int,float)) and not isinstance(value,bool) and np.isfinite(value):
            metric_records.append({'file':path.name, 'metric':prefix, 'value':float(value)})
all_metrics = pd.DataFrame(metric_records)
if len(all_metrics):
    signal_pattern = 'accuracy|f1|auc|precision|recall|r2|rmse|mae|loss|coverage|review|drift|psi|brier|calibration|revenue|cost|effect|lift|latency|row|reject|duplicate'
    decision_metrics = all_metrics[all_metrics['metric'].str.contains(signal_pattern, case=False, regex=True)].copy()
    if not len(decision_metrics):
        decision_metrics = all_metrics.copy()
    decision_metrics = decision_metrics.drop_duplicates(['file','metric']).reset_index(drop=True)
    display(decision_metrics.head(60).round(6))
    rate_like = decision_metrics[decision_metrics['metric'].str.contains('accuracy|f1|auc|precision|recall|coverage|rate|r2', case=False, regex=True)]
    if len(rate_like):
        bounded = rate_like[(rate_like['value']>=-1)&(rate_like['value']<=1)].head(30)
        if len(bounded):
            plt.figure(figsize=(10,max(5,0.3*len(bounded))))
            plt.barh(range(len(bounded)), bounded['value'])
            plt.yticks(range(len(bounded)), bounded['file']+' :: '+bounded['metric'])
            plt.xlim(min(-0.05,bounded['value'].min()-0.05),1.05)
            plt.title('Retained rate / quality metrics')
            plt.tight_layout()
            plt.show()
    error_like = decision_metrics[decision_metrics['metric'].str.contains('rmse|mae|loss|error|latency|drift|psi|brier', case=False, regex=True)]
    if len(error_like):
        display(error_like.sort_values('value', ascending=False).head(30).round(6))
else:
    print('No retained scalar JSON metrics are available yet.')


In [ ]:
# Inspect artifact sizes — a quick engineering sanity check
artifact_rows = []
for base in [PROJECT/'artifacts', PROJECT/'results', PROJECT/'outputs', ROOT/'verified'/PROJECT_SLUG]:
    if not base.exists():
        continue
    for path in base.rglob('*'):
        if path.is_file():
            artifact_rows.append({'file':str(path.relative_to(ROOT)) if ROOT in path.parents else str(path), 'suffix':path.suffix.lower(), 'size_kb':path.stat().st_size/1024})
artifacts_df = pd.DataFrame(artifact_rows).sort_values('size_kb', ascending=False) if artifact_rows else pd.DataFrame()
if len(artifacts_df):
    display(artifacts_df.head(40).round(2))
    by_type = artifacts_df.groupby('suffix', as_index=False).agg(files=('file','size'), total_kb=('size_kb','sum')).sort_values('total_kb', ascending=False)
    display(by_type.round(2))
    plt.figure(figsize=(8,4))
    plt.bar(by_type['suffix'].replace('', '<none>'), by_type['total_kb'])
    plt.ylabel('Total KB')
    plt.title('Retained evidence by file type')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('No retained artifacts/results found.')


In [ ]:
# Threshold / coverage trade-off when a result table contains confidence or probability
for path, table in result_tables:
    conf_cols = [c for c in table.columns if any(token in str(c).lower() for token in ('confidence','probability','proba','score','risk'))]
    correct_cols = [c for c in table.columns if 'correct' in str(c).lower()]
    if not conf_cols or not len(table):
        continue
    confidence = pd.to_numeric(table[conf_cols[0]], errors='coerce')
    valid_conf = confidence.notna()
    if valid_conf.sum() < 20:
        continue
    trade_rows = []
    for threshold in np.linspace(float(confidence[valid_conf].quantile(0.10)), float(confidence[valid_conf].quantile(0.90)), 9):
        accepted = valid_conf & (confidence >= threshold)
        row = {'threshold':float(threshold), 'coverage':float(accepted.mean()), 'review_rate':float((valid_conf & ~accepted).sum()/valid_conf.sum()), 'accepted_rows':int(accepted.sum())}
        if correct_cols:
            correctness = table[correct_cols[0]].astype(bool)
            row['accepted_accuracy'] = float(correctness[accepted].mean()) if accepted.any() else np.nan
        trade_rows.append(row)
    trade = pd.DataFrame(trade_rows)
    print('Trade-off table from', path.name, 'using', conf_cols[0])
    display(trade.round(4))
    plt.figure(figsize=(8,4))
    plt.plot(trade['threshold'], trade['coverage'], marker='o', label='coverage')
    if 'accepted_accuracy' in trade:
        plt.plot(trade['threshold'], trade['accepted_accuracy'], marker='o', label='accepted accuracy')
    plt.xlabel('Threshold')
    plt.ylabel('Rate')
    plt.title(f'Threshold trade-off — {path.name}')
    plt.legend()
    plt.tight_layout()
    plt.show()
    break


In [ ]:
# Produce a concise evidence-backed decision memo inside the notebook
project_summary = {
    'project': PROJECT_SLUG,
    'local_data_or_evidence_files': int(len(candidate_files)),
    'result_tables': int(len(result_tables)),
    'json_evidence_files': int(len(json_files)),
    'visual_evidence_files': int(len(png_files)),
    'has_tests': bool((PROJECT/'tests').exists() and any((PROJECT/'tests').rglob('test*.py'))),
    'has_readme': bool((PROJECT/'README.md').exists()),
}
if df is not None:
    project_summary.update({'inspected_rows':int(len(df)), 'inspected_columns':int(df.shape[1]), 'duplicate_rows':int(df.duplicated().sum()), 'missing_cells':int(df.isna().sum().sum())})
summary_table = pd.DataFrame({'item':list(project_summary.keys()), 'value':list(project_summary.values())})
display(summary_table)
print('DECISION PRINCIPLE')
print('1. Use the measured evidence above, not model complexity, to choose the final approach.')
print('2. Inspect the worst slices/failures before making a business or operational recommendation.')
print('3. Keep uncertain, novel or high-impact cases on a review/escalation path where appropriate.')
print('4. Treat the documented limitations as part of the solution, not as boilerplate.')


# Engineering appendix — canonical application source

The analysis and visual evidence come first. The cells below preserve additional canonical Python from this project for reviewers who want to inspect pipelines, APIs, tests, feature code, monitoring and reusable implementation details.


## Canonical source: `run.py`


In [ ]:
from __future__ import annotations

import io
import json
import math
import urllib.request
import zipfile
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import ParameterGrid
from xgboost import XGBRegressor

ROOT = Path(__file__).resolve().parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"
ARTIFACTS = ROOT / "artifacts"
for folder in (DATA, RESULTS, ARTIFACTS):
    folder.mkdir(exist_ok=True)

DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00275/Bike-Sharing-Dataset.zip"
RANDOM_STATE = 42
TARGET = "cnt"
LEAKAGE_COLUMNS = {"casual", "registered"}


@dataclass
class SplitSummary:
    train_rows: int
    validation_rows: int
    test_rows: int
    train_end: str
    validation_end: str
    test_end: str


@dataclass
class Metrics:
    mae: float
    rmse: float
    rmsle: float
    r2: float


def download_hourly_data(force: bool = False) -> Path:
    target = DATA / "hour.csv"
    if target.exists() and not force:
        return target
    with urllib.request.urlopen(DATA_URL, timeout=60) as response:
        payload = response.read()
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        member = next(name for name in archive.namelist() if name.endswith("hour.csv"))
        target.write_bytes(archive.read(member))
    return target


def load_data(path: Path | None = None) -> pd.DataFrame:
    path = path or download_hourly_data()
    df = pd.read_csv(path)
    df["dteday"] = pd.to_datetime(df["dteday"], errors="raise")
    df["timestamp"] = df["dteday"] + pd.to_timedelta(df["hr"], unit="h")
    return df.sort_values("timestamp").reset_index(drop=True)


def audit_data(df: pd.DataFrame) -> dict[str, Any]:
    required = {
        "instant", "dteday", "season", "yr", "mnth", "hr", "holiday", "weekday",
        "workingday", "weathersit", "temp", "atemp", "hum", "windspeed", "casual",
        "registered", "cnt", "timestamp",
    }
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    if df["timestamp"].duplicated().any():
        raise ValueError("Hourly timestamp should be unique")
    if (df[TARGET] < 0).any():
        raise ValueError("Demand cannot be negative")
    if not df["timestamp"].is_monotonic_increasing:
        raise ValueError("Rows must be chronologically ordered")
    return {
        "rows": int(len(df)),
        "columns": int(df.shape[1]),
        "missing_cells": int(df.isna().sum().sum()),
        "duplicate_timestamps": int(df["timestamp"].duplicated().sum()),
        "start": str(df["timestamp"].min()),
        "end": str(df["timestamp"].max()),
        "target_mean": float(df[TARGET].mean()),
        "target_median": float(df[TARGET].median()),
        "target_max": int(df[TARGET].max()),
    }


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["hour_sin"] = np.sin(2 * np.pi * out["hr"] / 24)
    out["hour_cos"] = np.cos(2 * np.pi * out["hr"] / 24)
    out["weekday_sin"] = np.sin(2 * np.pi * out["weekday"] / 7)
    out["weekday_cos"] = np.cos(2 * np.pi * out["weekday"] / 7)
    out["month_sin"] = np.sin(2 * np.pi * out["mnth"] / 12)
    out["month_cos"] = np.cos(2 * np.pi * out["mnth"] / 12)
    out["is_weekend"] = out["weekday"].isin([0, 6]).astype(int)
    out["is_commute_hour"] = out["hr"].isin([7, 8, 9, 16, 17, 18]).astype(int)
    out["feels_like_gap"] = out["atemp"] - out["temp"]
    out["weather_discomfort"] = out["hum"] * out["windspeed"]
    out["trend_hour"] = np.arange(len(out), dtype=float)
    out["days_from_start"] = (out["timestamp"] - out["timestamp"].min()).dt.total_seconds() / 86400.0
    return out


def chronological_split(df: pd.DataFrame, train_fraction: float = 0.70, validation_fraction: float = 0.15):
    n = len(df)
    train_end = int(n * train_fraction)
    validation_end = int(n * (train_fraction + validation_fraction))
    train = df.iloc[:train_end].copy()
    validation = df.iloc[train_end:validation_end].copy()
    test = df.iloc[validation_end:].copy()
    if not (train["timestamp"].max() < validation["timestamp"].min() <= validation["timestamp"].max() < test["timestamp"].min()):
        raise ValueError("Chronological split is not strictly ordered")
    summary = SplitSummary(
        train_rows=len(train),
        validation_rows=len(validation),
        test_rows=len(test),
        train_end=str(train["timestamp"].max()),
        validation_end=str(validation["timestamp"].max()),
        test_end=str(test["timestamp"].max()),
    )
    return train, validation, test, summary


def feature_columns(df: pd.DataFrame) -> list[str]:
    excluded = {
        TARGET,
        "instant",
        "dteday",
        "timestamp",
        *LEAKAGE_COLUMNS,
    }
    features = [c for c in df.columns if c not in excluded]
    if LEAKAGE_COLUMNS.intersection(features):
        raise AssertionError("Leakage columns entered model features")
    return features


def make_xy(df: pd.DataFrame, features: list[str]) -> tuple[pd.DataFrame, pd.Series]:
    return df[features].copy(), df[TARGET].astype(float).copy()


def metrics(y_true: pd.Series | np.ndarray, y_pred: np.ndarray) -> Metrics:
    truth = np.asarray(y_true, dtype=float)
    pred = np.maximum(np.asarray(y_pred, dtype=float), 0.0)
    return Metrics(
        mae=float(mean_absolute_error(truth, pred)),
        rmse=float(math.sqrt(mean_squared_error(truth, pred))),
        rmsle=float(math.sqrt(mean_squared_error(np.log1p(truth), np.log1p(pred)))),
        r2=float(r2_score(truth, pred)),
    )


def seasonal_hour_baseline(train: pd.DataFrame, target: pd.DataFrame) -> np.ndarray:
    lookup = train.groupby(["weekday", "hr"])[TARGET].median()
    global_median = float(train[TARGET].median())
    preds = []
    for row in target[["weekday", "hr"]].itertuples(index=False):
        preds.append(float(lookup.get((row.weekday, row.hr), global_median)))
    return np.asarray(preds)


def model_from_params(params: dict[str, Any]) -> XGBRegressor:
    return XGBRegressor(
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist",
        eval_metric="mae",
        **params,
    )


def tune_xgboost(train: pd.DataFrame, validation: pd.DataFrame, features: list[str]):
    x_train, y_train = make_xy(train, features)
    x_val, y_val = make_xy(validation, features)
    grid = ParameterGrid(
        {
            "n_estimators": [300, 600],
            "max_depth": [4, 7],
            "learning_rate": [0.03, 0.07],
            "subsample": [0.8],
            "colsample_bytree": [0.8],
            "min_child_weight": [1, 5],
            "reg_lambda": [1.0],
        }
    )
    trials: list[dict[str, Any]] = []
    best_model: XGBRegressor | None = None
    best_mae = float("inf")
    best_params: dict[str, Any] = {}

    for params in grid:
        model = model_from_params(params)
        model.fit(x_train, y_train, verbose=False)
        pred = model.predict(x_val)
        score = metrics(y_val, pred)
        trial = {**params, **asdict(score)}
        trials.append(trial)
        if score.mae < best_mae:
            best_mae = score.mae
            best_model = model
            best_params = dict(params)

    if best_model is None:
        raise RuntimeError("No XGBoost model trained")
    return best_model, best_params, pd.DataFrame(trials).sort_values("mae")


def refit_best(train: pd.DataFrame, validation: pd.DataFrame, features: list[str], params: dict[str, Any]) -> XGBRegressor:
    combined = pd.concat([train, validation], axis=0).sort_values("timestamp")
    x, y = make_xy(combined, features)
    model = model_from_params(params)
    model.fit(x, y, verbose=False)
    return model


def error_slices(test: pd.DataFrame, prediction: np.ndarray) -> pd.DataFrame:
    work = test[["timestamp", "hr", "weekday", "weathersit", TARGET]].copy()
    work["prediction"] = np.maximum(prediction, 0.0)
    work["absolute_error"] = np.abs(work[TARGET] - work["prediction"])
    work["demand_band"] = pd.cut(work[TARGET], bins=[-1, 50, 150, 300, np.inf], labels=["low", "medium", "high", "very_high"])
    slices = []
    for column in ["demand_band", "weathersit", "hr"]:
        grouped = work.groupby(column, observed=True).agg(
            rows=(TARGET, "size"),
            actual_mean=(TARGET, "mean"),
            prediction_mean=("prediction", "mean"),
            mae=("absolute_error", "mean"),
        ).reset_index()
        grouped.insert(0, "slice", column)
        grouped.rename(columns={column: "value"}, inplace=True)
        slices.append(grouped)
    return pd.concat(slices, ignore_index=True)


def feature_importance_table(model: XGBRegressor, features: list[str]) -> pd.DataFrame:
    values = model.feature_importances_
    return pd.DataFrame({"feature": features, "importance": values}).sort_values("importance", ascending=False)


def operational_decisions(prediction: np.ndarray, residual_mae: float) -> pd.DataFrame:
    pred = np.maximum(prediction, 0.0)
    lower = np.maximum(pred - 1.5 * residual_mae, 0.0)
    upper = pred + 1.5 * residual_mae
    band = np.select(
        [pred >= 400, pred >= 250, pred >= 120],
        ["critical", "high", "moderate"],
        default="normal",
    )
    action = np.select(
        [pred >= 400, pred >= 250, pred >= 120],
        ["pre-position bikes and staff", "rebalance inventory", "monitor capacity"],
        default="standard operations",
    )
    return pd.DataFrame(
        {
            "predicted_demand": pred,
            "uncertainty_lower": lower,
            "uncertainty_upper": upper,
            "demand_risk_band": band,
            "recommended_action": action,
        }
    )


def predict_next(model: XGBRegressor, row: pd.DataFrame, features: list[str], validation_mae: float) -> dict[str, Any]:
    value = float(max(model.predict(row[features])[0], 0.0))
    if value >= 400:
        band, action = "critical", "pre-position bikes and staff"
    elif value >= 250:
        band, action = "high", "rebalance inventory"
    elif value >= 120:
        band, action = "moderate", "monitor capacity"
    else:
        band, action = "normal", "standard operations"
    return {
        "predicted_hourly_demand": value,
        "uncertainty_proxy": [max(0.0, value - 1.5 * validation_mae), value + 1.5 * validation_mae],
        "demand_risk_band": band,
        "recommended_action": action,
    }


def save_json(path: Path, payload: Any) -> None:
    path.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")


def main() -> None:
    raw = load_data()
    audit = audit_data(raw)
    data = add_features(raw)
    train, validation, test, split_summary = chronological_split(data)
    features = feature_columns(data)

    baseline_pred = seasonal_hour_baseline(train, validation)
    baseline_metrics = metrics(validation[TARGET], baseline_pred)

    _, best_params, trials = tune_xgboost(train, validation, features)
    validation_model = model_from_params(best_params)
    x_train, y_train = make_xy(train, features)
    x_validation, y_validation = make_xy(validation, features)
    validation_model.fit(x_train, y_train, verbose=False)
    validation_pred = validation_model.predict(x_validation)
    validation_metrics = metrics(y_validation, validation_pred)

    final_model = refit_best(train, validation, features, best_params)
    x_test, y_test = make_xy(test, features)
    test_pred = final_model.predict(x_test)
    test_metrics = metrics(y_test, test_pred)

    slices = error_slices(test, test_pred)
    importance = feature_importance_table(final_model, features)
    decisions = operational_decisions(test_pred, validation_metrics.mae)
    predictions = test[["timestamp", TARGET]].reset_index(drop=True).join(decisions)

    trials.to_csv(RESULTS / "tuning_trials.csv", index=False)
    slices.to_csv(RESULTS / "error_slices.csv", index=False)
    importance.to_csv(RESULTS / "feature_importance.csv", index=False)
    predictions.to_csv(RESULTS / "test_predictions.csv", index=False)
    joblib.dump({"model": final_model, "features": features, "validation_mae": validation_metrics.mae}, ARTIFACTS / "xgboost_bike_demand.joblib")

    bundle = joblib.load(ARTIFACTS / "xgboost_bike_demand.joblib")
    parity = np.allclose(bundle["model"].predict(x_test.iloc[:20]), final_model.predict(x_test.iloc[:20]))
    if not parity:
        raise RuntimeError("Saved model parity check failed")

    payload = {
        "dataset_audit": audit,
        "split": asdict(split_summary),
        "features": features,
        "best_parameters": best_params,
        "validation_baseline": asdict(baseline_metrics),
        "validation_xgboost": asdict(validation_metrics),
        "test_xgboost": asdict(test_metrics),
        "validation_mae_gain_vs_baseline": float(baseline_metrics.mae - validation_metrics.mae),
        "top_features": importance.head(12).to_dict(orient="records"),
        "example_decision": predict_next(final_model, test.iloc[[0]], features, validation_metrics.mae),
        "limitations": [
            "Historical location-specific data; no live station inventory or current road conditions.",
            "Residual-based uncertainty proxy is not a calibrated predictive interval.",
            "Operational thresholds are illustrative and require real capacity-cost calibration.",
        ],
    }
    save_json(RESULTS / "metrics.json", payload)
    print(json.dumps(payload, indent=2, default=str))


if __name__ == "__main__":
    main()


## Canonical source: `tests/test_xgboost.py`


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(PROJECT))

from run import LEAKAGE_COLUMNS, TARGET, add_features, chronological_split, feature_columns, metrics


def sample_frame(rows: int = 100):
    timestamps = pd.date_range("2024-01-01", periods=rows, freq="h")
    frame = pd.DataFrame({
        "instant": np.arange(rows),
        "dteday": timestamps.normalize(),
        "season": 1,
        "yr": 0,
        "mnth": timestamps.month,
        "hr": timestamps.hour,
        "holiday": 0,
        "weekday": timestamps.dayofweek,
        "workingday": 1,
        "weathersit": 1,
        "temp": 0.5,
        "atemp": 0.5,
        "hum": 0.5,
        "windspeed": 0.2,
        "casual": 10,
        "registered": 20,
        "cnt": 30,
        "timestamp": timestamps,
    })
    return add_features(frame)


def test_leakage_columns_are_excluded():
    frame = sample_frame()
    features = feature_columns(frame)
    assert TARGET not in features
    assert not LEAKAGE_COLUMNS.intersection(features)


def test_chronological_split_is_ordered():
    train, validation, test, _ = chronological_split(sample_frame(200))
    assert train.timestamp.max() < validation.timestamp.min()
    assert validation.timestamp.max() < test.timestamp.min()


def test_metrics_are_zero_for_perfect_prediction():
    y = np.array([1.0, 4.0, 9.0])
    result = metrics(y, y)
    assert result.mae == 0.0
    assert result.rmse == 0.0


# Portfolio depth check

**Meaningful visible code lines after all notebook passes:** 932. The working target for a major application is roughly 1,000 meaningful lines when justified by the problem. This notebook is in/above the working depth range. Line count is never permission to add filler; depth must come from data, analysis, visualisation, modelling/engineering, evaluation, robustness and decision logic.
